# 07 - NASA C-MAPSS Early-Warning Multi-Horizon Classification
### Phase 2: Machine Learning Baseline Development & Evaluation

This notebook evaluates binary classification models across three critical early-warning horizons:
- **Horizon 10** ($RUL \le 10$ cycles): Emergency maintenance alert.
- **Horizon 20** ($RUL \le 20$ cycles): Tactical dispatch alert.
- **Horizon 30** ($RUL \le 30$ cycles): Advance supply-chain / part scheduling.

Decision thresholds are strictly tuned on validation data to optimize F1 / Recall under class imbalance.


In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root / "src"))

from evaluation.metrics import compute_classification_metrics, find_optimal_threshold
from training.nasa_rul_trainer import load_nasa_tabular_data


## 1. Load Tabular Data & Class Distributions


In [ ]:
proc_dir = project_root / "data" / "processed" / "nasa"
train_df, val_df, test_df, feature_cols = load_nasa_tabular_data("FD001", processed_dir=proc_dir)

for h in [10, 20, 30]:
    rate = train_df[f"early_warning_{h}"].mean() * 100
    print(f"Horizon H={h} Positive Class Prevalence (Train): {rate:.2f}%")


## 2. Load Models & Evaluate Test Performance Across Horizons


In [ ]:
models_dir = project_root / "models" / "baseline"
models = ["Logistic Regression", "Random Forest Classifier", "Gradient Boosting Classifier", "XGBoost Classifier"]

results = []
X_val = val_df[feature_cols].to_numpy(dtype=np.float32)
X_test = test_df[feature_cols].to_numpy(dtype=np.float32)

for h in [10, 20, 30]:
    y_val = val_df[f"early_warning_{h}"].to_numpy(dtype=np.int32)
    y_test = test_df[f"early_warning_{h}"].to_numpy(dtype=np.int32)
    
    for m in models:
        clean_m = m.lower().replace(" ", "_")
        path = models_dir / f"nasa_ew_{h}_{clean_m}_FD001.joblib"
        if path.exists():
            clf = joblib.load(path)
            y_val_prob = clf.predict_proba(X_val)[:, 1] if hasattr(clf, "predict_proba") else 1.0 / (1.0 + np.exp(-clf.decision_function(X_val)))
            y_test_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else 1.0 / (1.0 + np.exp(-clf.decision_function(X_test)))
            
            opt_th, _ = find_optimal_threshold(y_val, y_val_prob)
            val_m = compute_classification_metrics(y_val, y_val_prob, threshold=opt_th, prefix="val")
            test_m = compute_classification_metrics(y_test, y_test_prob, threshold=opt_th, prefix="test")
            
            results.append({
                "Horizon": f"H={h}",
                "Model": m,
                "Optimal Threshold": opt_th,
                "Val F1": val_m["val_f1"],
                "Val PR-AUC": val_m["val_pr_auc"],
                "Test F1": test_m["test_f1"],
                "Test PR-AUC": test_m["test_pr_auc"],
                "Test Recall": test_m["test_recall"],
                "Test Precision": test_m["test_precision"],
            })

df_ew_results = pd.DataFrame(results)
display(df_ew_results)
